# Deep Learning Framework for Environmental Sound Recognition Using PyTorch
### Notebook 02 — Stage 4 controlled comparison and Stage 5 held-out evaluation

**Current architecture:** Official AudioSet-pretrained PANNs CNN14 · **Dataset:** UrbanSound8K · **Task:** single-label ten-class classification.

This notebook builds on the Stage 1–4 laboratory notebook. It **does not replace or erase** the previously executed notebook. It reviews three actual complete Stage 4 training runs, selects exactly one model **based on validation evidence**, and provides a deliberately opt-in **one-time** held-out test evaluation for fold 1.

> **Evidence status at creation:** Stage 4 full-fold training results have been inspected and tabulated from the ZIP shared on 27 October 2026. `block6` is preselected based on validation macro-F1. The Stage 5 test has **not** yet been run; no test results are supplied or simulated. Keep the test-fold evaluation switch off until the selected checkpoint directory is confirmed.


## 1 · Safe controls

Running **Restart & Run All** with the default settings reviews existing Stage 4 results only. It will not train, install libraries, reclone repositories, delete caches, or evaluate test audio.

**Do not evaluate multiple configurations on test fold 1.** The choice of `block6` was made from validation fold 2 in advance. The test fold is for one final unbiased evaluation, not hyperparameter selection.


In [ ]:
RUN_HELDOUT_TEST = False   # Set True only after selecting a verified block6 best.pt
UPDATE_ROOT_README = False  # Set True to insert/update the evidence section, preserving other text

# Replace None with the actual Stage 4 run directory containing block6 best.pt
# Example: r"D:\EnvironmentalSoundRecognition\results\stage4\20261027_123456_123456_block6_test1"
SELECTED_BLOCK6_RUN_DIR = None

# Real inference on 873 test files; batch 1 is conservative for the 4 GiB RTX 3050.
TEST_BATCH_SIZE = 1
TEST_AMP_MODE = 'auto'  # BF16 on supported CUDA GPUs, otherwise FP32
TEST_DEVICE = 'auto'

print('Test evaluation enabled:', RUN_HELDOUT_TEST)
print('README synchronization enabled:', UPDATE_ROOT_README)

## 2 · Project discovery and reproducibility

Notebook code finds the project root whether VS Code uses the repository root or the `notebooks/` folder as its working directory. Use the same Python interpreter as Stage 4 (`torch==2.7.0+cu128` on your PC). The provided comparison CSVs are **derived only from your uploaded Stage 4 histories**.


In [ ]:
from pathlib import Path
import sys, json, os, subprocess, datetime
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'src'/'dataset.py').is_file() and (p/'reports'/'stage4_comparison'/'stage4_comparison.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the project root after extracting the Stage5 kit.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
REVIEW = ROOT/'reports'/'stage4_comparison'
print('Project root:', ROOT)
print('Python kernel:', sys.executable)
print('Stage 4 analysis directory:', REVIEW)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())

## 3 · CO5 controlled ablation — actual validation results

**Only validation fold 2 was scored during model development; test fold 1 remains reserved.** The experiments are directly comparable by data partitions, seed, preprocessing, training budget, precision, and optimizer settings, but only one random seed and one outer test-fold setting have been used.


In [ ]:
comparison = pd.read_csv(REVIEW/'stage4_comparison.csv')
assert set(comparison.configuration) == {'head', 'fc1', 'block6'}
assert (comparison.best_epoch == 4).all()
assert (~comparison.test_evaluated.astype(bool)).all()
columns = ['configuration','best_epoch','best_val_accuracy_pct','best_val_macro_f1',
           'trainable_parameters','total_six_epoch_minutes','peak_gpu_mib']
display(comparison[columns].set_index('configuration'))
head_row = comparison.set_index('configuration').loc['head']
block_row = comparison.set_index('configuration').loc['block6']
print(f"Block6 – head: {block_row.best_val_accuracy_pct-head_row.best_val_accuracy_pct:.2f} percentage points accuracy")
print(f"Block6 – head: {block_row.best_val_macro_f1-head_row.best_val_macro_f1:.4f} macro-F1")
print('FINAL TEST STATUS: NOT EVALUATED')

In [ ]:
display(Image(filename=str(REVIEW/'validation_comparison.png'), width=850))

## 4 · Convergence, learning curves, and overfitting

All three runs selected epoch 4 using validation macro-F1. Validation loss increases after epoch 4 even as training loss falls, most clearly for the `block6` run. This favors evaluating the **validation-selected `best.pt`**, not the final epoch.

**Important:** training accuracy is measured online with SpecAugmentation/dropout enabled, while validation uses evaluation mode. Direct subtraction is **not** a clean train–validation generalization gap estimate. The change in validation loss/F1 across epochs is more informative here.


In [ ]:
all_epochs = pd.read_csv(REVIEW/'stage4_all_epochs.csv')
selected = all_epochs.loc[all_epochs.epoch==4, ['configuration','epoch','train_loss','train_accuracy','val_loss','val_accuracy','val_macro_f1','optimizer_steps']]
display(selected.set_index('configuration'))
display(Image(filename=str(REVIEW/'macro_f1_by_epoch.png'), width=820))
display(Image(filename=str(REVIEW/'val_loss_by_epoch.png'), width=820))

## 5 · Freeze-depth cost and interpretation

The `block6` configuration fine-tunes 60,848,138 parameters (about 74.3% of the 81,857,561 parameters), compared with only 20,490 parameters for head-only. Its observed maximum allocated GPU memory was 1,278.1 MiB and six-epoch runtime about 28.65 minutes. These are **observed run measurements**, not theoretical architecture complexity or latency benchmarks.

The validation improvement is encouraging but **does not establish multi-fold stability or statistical significance**. Five `fsID` source IDs span official folds, including some overlapping annotated intervals. We preserve the original folds, document the caveat, and avoid calling them fully source-independent.


In [ ]:
display(Image(filename=str(REVIEW/'train_accuracy_by_epoch.png'), width=820))
print((REVIEW/'STAGE4_REVIEW.md').read_text(encoding='utf-8')[:4500])

## 6 · Select the single held-out-test candidate (from validation only)

**Preselected model:** `block6`, epoch 4, validation macro-F1 = **0.9009665073**, validation accuracy = **0.9065315315**. Its full Stage 4 run should have `diagnostic_only: false`, `test_evaluated: false`, and a `best.pt` file.

The ZIP you provided contained metrics and plots but **not the checkpoints or original run-directory names**. Therefore, the next cell lists actual run directories on *your PC* rather than inventing a path.


In [ ]:
STAGE4_DIR = ROOT/'results'/'stage4'
matching = []
if STAGE4_DIR.is_dir():
    for folder in sorted(STAGE4_DIR.iterdir()):
        if not folder.is_dir(): continue
        cfg_file, summary_file = folder/'config.json', folder/'summary.json'
        if not cfg_file.exists() or not summary_file.exists(): continue
        try:
            cfg, summ = json.loads(cfg_file.read_text()), json.loads(summary_file.read_text())
        except (ValueError, OSError): continue
        if (cfg.get('config') == 'block6' and cfg.get('test_fold') == 1 and
            cfg.get('diagnostic_only') is False and summ.get('diagnostic_only') is False and
            summ.get('best_epoch') == 4 and
            abs(summ.get('best_val_macro_f1',-999)-0.9009665073491062) < 1e-8 and
            (folder/'best.pt').exists()):
            matching.append(folder)
if matching:
    print('Validation-matching full block6 checkpoint directories:')
    for f in matching: print(' ',f)
else:
    print('No local matching block6 run detected; check Stage 4 results folder.')
print('Set SELECTED_BLOCK6_RUN_DIR in the controls cell to ONE of these exact paths before test.')

## 7 · One-time held-out test inference (Stage 5; manually enabled)

Evaluation loads the official AudioSet CNN14 state dictionary with strict key matching and applies the `best.pt` **trainable-parameter subset** from Stage 4. It then feeds only **test fold 1 (873 WAV files)** through `model.eval()` and computes accuracy, macro/weighted precision/recall/F1, per-class sensitivity/specificity, confusion matrices, and one-vs-rest ROC-AUC from probabilities.

This cell is **off by default**. Only execute after the previous selection cell finds and verifies the expected `block6` run directory. The evaluation script rejects pilot checkpoints and mismatched folds, and refuses to evaluate the same checkpoint again if a completed evaluation report already exists. **Do not use final test metrics to tune models.**


In [ ]:
if RUN_HELDOUT_TEST:
    if SELECTED_BLOCK6_RUN_DIR is None:
        raise ValueError('Set SELECTED_BLOCK6_RUN_DIR first in the controls cell.')
    selected_path = Path(SELECTED_BLOCK6_RUN_DIR).expanduser().resolve()
    if selected_path not in matching:
        raise ValueError('Selected folder is not one of the validated Stage 4 block6 runs shown above.')
    from src.evaluate_selected import validate_selected_run
    cfg, summ, ckpt = validate_selected_run(selected_path)
    print('ONE-TIME EVALUATION CONFIRMED:', selected_path)
    print('Validation selection: block6, epoch', summ['best_epoch'], 'macro-F1', summ['best_val_macro_f1'])
    command = [sys.executable, '-m', 'src.evaluate_selected', '--run-dir', str(selected_path),
               '--device', TEST_DEVICE, '--amp', TEST_AMP_MODE, '--batch-size', str(TEST_BATCH_SIZE)]
    logs = ROOT/'results'/'notebook_evidence'
    logs.mkdir(parents=True, exist_ok=True)
    logfile = logs/('heldout_test_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S') + '.log')
    print('Persistent log:', logfile)
    with logfile.open('w', encoding='utf-8') as out:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            out.write(line)
            out.flush()
            print(line, end='', flush=True)
        rc = process.wait()
    if rc != 0:
        raise RuntimeError(f'Held-out evaluation failed with exit code {rc}; inspect {logfile}')
    print('Held-out inference complete; do NOT repeat/tune from test fold.')
else:
    print('HELD-OUT TEST NOT RUN (default). Review Stage 4 evidence, then explicitly opt in once.')

## 8 · Read the genuine Stage 5 artifacts after execution

This cell reads saved evaluation results. It never launches test inference. When a Stage 5 run succeeds, record its test metrics as **fold-1** evidence, not ten-fold mean accuracy.


In [ ]:
STAGE5_DIR = ROOT/'results'/'stage5'
completed = []
if STAGE5_DIR.is_dir():
    for fn in STAGE5_DIR.glob('*/evaluation_summary.json'):
        try:
            result = json.loads(fn.read_text(encoding='utf-8'))
            if result.get('test_evaluated') is True and result.get('config') == 'block6':
                completed.append((fn.parent,result))
        except (OSError,ValueError): pass
if completed:
    for folder,result in sorted(completed, key=lambda x: str(x[0])):
        print('Selected evaluation:', folder)
        display(pd.DataFrame([{
            'test_fold': result['test_fold'], 'n_test': result['test_samples'],
            'accuracy':result['accuracy'],'macro_f1':result['macro_f1'],
            'macro_precision':result['macro_precision'],'macro_recall':result['macro_recall'],
            'macro_specificity_ovr':result['macro_specificity_ovr'],
            'macro_roc_auc_ovr':result['macro_roc_auc_ovr']}]))
        display(pd.read_csv(folder/'per_class_metrics.csv'))
        display(Image(filename=str(folder/'confusion_matrix.png'),width=850))
else:
    print('Stage 5 not evaluated yet. This is expected before the one-time test step.')

## 9 · Update root README without erasing your notebook diary

This cell safely inserts or replaces **only one bounded, clearly labeled section** of the root README. Existing historical notes and notebook diary sections are preserved. It includes actual Stage 4 validation metrics and Stage 5 metrics **only if the evaluation output exists**.

Set `UPDATE_ROOT_README = True` in controls to enable. Save the notebook (`Ctrl+S`) after execution.


In [ ]:
if UPDATE_ROOT_README:
    root_readme = ROOT/'README.md'
    start_marker = '<!-- BEGIN AUTOMATED STAGE4_STAGE5_EVIDENCE -->'
    end_marker = '<!-- END AUTOMATED STAGE4_STAGE5_EVIDENCE -->'
    stage5_lines = ['- **Stage 5 held-out test:** pending (test fold 1 not evaluated in this notebook session).']
    if completed:
        folder, evaluated = sorted(completed,key=lambda x:str(x[0]))[-1]
        stage5_lines = [
            f"- **Stage 5 held-out fold {evaluated['test_fold']}**: accuracy **{evaluated['accuracy']:.2%}**, "
            f"macro-F1 **{evaluated['macro_f1']:.4f}**, macro ROC-AUC (OvR) **{evaluated['macro_roc_auc_ovr']}**.",
            f"- Evidence: `{folder.relative_to(ROOT) / 'evaluation_summary.json'}`; test metrics are *not* used for tuning.",
        ]
    content = '\n'.join([
       start_marker,
       '## Latest verified experiments (27 October 2026)',
       '- Three genuine Stage 4 full-fold runs completed: train folds 3–10; validation fold 2; test fold 1 held aside.',
       '- Best validation checkpoint is epoch 4 for all configurations, chosen by macro-F1.',
       '- **head:** validation accuracy **87.73%**, macro-F1 **0.8638**, 20,490 trainable parameters.',
       '- **fc1:** validation accuracy **88.06%**, macro-F1 **0.8668**, 4,216,842 trainable parameters.',
       '- **block6:** validation accuracy **90.65%**, macro-F1 **0.9010**, 60,848,138 trainable parameters.',
       '- The selected model for one-time held-out evaluation is **block6**, from validation fold 2; it is NOT multi-fold performance.',
       '- Validation loss rises after epoch 4; online training and evaluation accuracy are not directly interchangeable.',
       '- Five Freesound source IDs span official folds; this source-isolation caveat remains disclosed.',
       *stage5_lines,
       '- Detailed metrics and plots: `reports/stage4_comparison/`.',
       end_marker,
    ])
    original = root_readme.read_text(encoding='utf-8') if root_readme.is_file() else '# Deep Learning Framework for Environmental Sound Recognition Using PyTorch\n'
    if start_marker in original and end_marker in original:
        a=original.index(start_marker)
        b=original.index(end_marker)+len(end_marker)
        updated = original[:a]+content+original[b:]
    else:
        # Insert the newest status directly under the project title, keeping
        # older historical sections and user-authored diary text unchanged.
        first_line_end = original.find('\n')
        if first_line_end < 0:
            updated = original.rstrip()+'\n\n'+content+'\n'
        else:
            updated = original[:first_line_end+1]+'\n'+content+'\n\n'+original[first_line_end+1:]
    root_readme.write_text(updated,encoding='utf-8')
    print('Updated bounded evidence section only:',root_readme)
else:
    print('README left unchanged (set UPDATE_ROOT_README=True to synchronize).')

## 10 · Next deliverables: CO3 / CO4 / CO5

- **CO3:** Once the one-time test produces a real held-out evaluation, explain accuracy, precision, recall, macro-F1, per-class breakdown, confusion matrix, specificity and multiclass ROC-AUC. Keep the exact fold number attached to every score.
- **CO4:** Build a cited comparison against verifiable UrbanSound8K studies, explicitly distinguishing the papers' cross-validation protocols, AudioSet pretraining and data augmentation from ours. Do not compare different protocols as if equal.
- **CO5:** The head/fc1/block6 study is an ablation with one seed and one validation fold. Multi-fold stability or repeat-seed tests are a separate, future controlled study; they have **not** been completed.
- **Reproducibility and submission:** keep saved `best.pt` files on your PC, `requirements.lock.txt`, official PANNs commit, notebooks, README, and test result artifacts. Do not add WAVs or ~1.4 GB checkpoints to Git by accident. The `.gitignore` delivered earlier should remain in the project root.

**Stop after the one-time selected-model test and inspect the report before implementing any additional experiment.** Do not choose a different architecture based on test-fold performance.
